# NVIDIA Nemotron Model Reasoning Challenge — Train LoRA + Build `submission.zip`

This notebook is **self-contained**: it clones the project repo, installs it, fine-tunes a
LoRA adapter on the competition data, and produces `submission.zip` (the LoRA adapter the
competition grades server-side via vLLM).

## Requirements (set these in the Kaggle notebook UI before running)
- **Competition data attached**: add the *NVIDIA Nemotron Model Reasoning Challenge* dataset
  via **Add Input -> Competitions**. It mounts at `/kaggle/input/nvidia-nemotron-model-reasoning-challenge/`.
- **GPU accelerator ON**: Settings -> Accelerator -> GPU.
- **Internet ON**: Settings -> Internet (needed to clone the repo, `pip install`, and pull
  the base model from Hugging Face).

## Which GPU / which training config
Kaggle gives you a few GPU options. Pick the matching Hydra `train=` preset:

| Kaggle accelerator | Preset | Why |
|---|---|---|
| **T4 x2** or **P100** (16 GB) | `train=qlora_t4` | 4-bit QLoRA, rank 16 — fits 16 GB |
| Bigger GPU (e.g. A100 40/80 GB, if available) | `train=lora_a100` | bf16 LoRA, rank 32, larger batch |

Default in this notebook is **`qlora_t4`** since T4 x2 is the common free Kaggle GPU.
LoRA rank is capped at **32** (competition rule) and both presets respect that.

## Grading recap
The server loads your LoRA adapter into vLLM and scores generated answers. The model must
emit reasoning inside `<think>...</think>` and the final answer inside `\boxed{...}`.

## 1. Clone the repo and install

Kaggle uses **pip** (not `uv`). We install the project editable with the `[gpu]` extra
(`bitsandbytes` + `accelerate`) so 4-bit QLoRA works. `transformers` is pinned `>=4.45,<5`.

`uv` is optional — if you prefer it you can `pip install uv && uv pip install -e ".[gpu]"`,
but plain `pip` is fine here.

**Unsloth is optional** (`pip install unsloth`) for faster/leaner training; the code falls
back to PEFT/TRL automatically when Unsloth is not installed, so you can skip it.

In [ ]:
# Clone (idempotent: skip if already cloned in this session)
import os

REPO_DIR = "/kaggle/working/nemotron-reasoning-challenge"
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/charleneleong-ai/nemotron-reasoning-challenge.git {REPO_DIR}
%cd {REPO_DIR}
# Use the branch with the Kaggle run wiring (change/remove if merged to main)
!git checkout feat/kaggle-real-run || echo "branch not found; staying on default branch"

In [ ]:
# Install the project + GPU extra. Pin transformers to a 4.x that supports trust_remote_code.
!pip install -q -e ".[gpu]"
!pip install -q "transformers>=4.45,<5"

# OPTIONAL — Unsloth for faster QLoRA. The repo falls back to PEFT/TRL if this is absent.
# !pip install -q unsloth

# The custom-code Nemotron base model is a Mamba-MoE that needs trust_remote_code kernels.
# These usually build on Kaggle GPU images; uncomment if the train step complains about them.
# !pip install -q mamba-ssm causal-conv1d

## 2. Point the pipeline at the attached competition data

The competition data mounts read-only at:

```
/kaggle/input/nvidia-nemotron-model-reasoning-challenge/
    train.csv
    test.csv
```

Do **not** run `main download` inside Kaggle — that path uses the Kaggle API / `.env`
credentials, which differ here. Instead use the **attached input** directly.

Two equivalent options:
1. Override the Hydra data path on the CLI: `data.path=/kaggle/input/.../train.csv` (used below).
2. Or symlink the input into `data/` so the default config finds it.

The cell below lists what's actually mounted and sets `TRAIN_CSV` accordingly.

In [ ]:
import glob
import os

COMP = "nvidia-nemotron-model-reasoning-challenge"
INPUT_DIR = f"/kaggle/input/{COMP}"

print("Mounted competition files:")
for p in sorted(glob.glob(f"{INPUT_DIR}/**/*", recursive=True)):
    print(" ", p)

# Resolve train.csv (it may sit at the top level or one dir down)
cands = glob.glob(f"{INPUT_DIR}/**/train.csv", recursive=True)
assert cands, (
    f"train.csv not found under {INPUT_DIR} — is the competition attached as an input?"
)
TRAIN_CSV = cands[0]
TEST_CSV = (glob.glob(f"{INPUT_DIR}/**/test.csv", recursive=True) or [None])[0]
print("\nTRAIN_CSV =", TRAIN_CSV)
print("TEST_CSV  =", TEST_CSV)

# Option 2 (optional): symlink into data/ so the default config path works too.
os.makedirs("data", exist_ok=True)
if not os.path.exists("data/train.csv"):
    os.symlink(TRAIN_CSV, "data/train.csv")

## 3. Hugging Face auth (base model may be gated)

The base model (`nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16`) may be **gated** on
Hugging Face. Add your token as a **Kaggle Secret**:

> Kaggle notebook -> **Add-ons -> Secrets -> Add a new secret**, name it `HF_TOKEN`,
> paste your `hf_...` token, and enable it for this notebook.

The cell logs in if the secret is present and skips quietly otherwise (e.g. if the model is
ungated by the time you run).

In [ ]:
# Log in to Hugging Face using a Kaggle Secret named HF_TOKEN (add it via Add-ons -> Secrets).
try:
    from kaggle_secrets import UserSecretsClient

    token = UserSecretsClient().get_secret("HF_TOKEN")
    from huggingface_hub import login

    login(token=token)
    os.environ["HF_TOKEN"] = token  # some loaders read the env var
    print("Hugging Face login OK")
except Exception as e:
    print(f"Skipping HF login ({e}). Fine if the base model is ungated.")

## 4. (Optional) Inspect target modules to pick concrete `lora_target_modules`

The default config uses `lora_target_modules: all-linear`, which works out of the box. If you
want to target a concrete subset (smaller adapter, faster training), run the helper script — it
materializes the architecture on the **meta device** (no ~60 GB download) and prints the unique
`nn.Linear` suffixes plus a ready-to-paste YAML line.

This needs the custom Nemotron kernels importable on a CUDA box, so it can fail on some images;
it's purely optional — skip it and keep `all-linear`.

In [ ]:
# Optional: list nn.Linear suffixes you could target instead of all-linear.
!python scripts/inspect_target_modules.py --model nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16 || echo "inspect skipped (kernels/auth unavailable) — keep all-linear"

## 5. Train the LoRA adapter

Runs the project CLI via Hydra overrides. We use **`train=qlora_t4`** (4-bit, fits T4/P100)
and feed the attached competition CSV through `data.path`.

Notes:
- This downloads the base model on first run (large) and then fine-tunes — expect a long
  runtime; keep within Kaggle's session limit (~9–12 h GPU). Reduce work with
  `train.max_steps=...` or `data.max_samples=...` for a quick smoke run first.
- If you have a **bigger GPU**, swap to `train=lora_a100` (bf16, rank 32, larger batch).
- The adapter is written to `cfg.train.output_dir` (`adapters/qlora_t4` for this preset).

In [ ]:
# Quick smoke run first (recommended): tiny step budget to validate the pipeline end-to-end.
# !python -m src.main train model=nemotron_nano train=qlora_t4 data.path={TRAIN_CSV} train.max_steps=5 data.max_samples=50

# Full training run (qlora_t4 = 4-bit QLoRA, fits T4 x2 / P100):
!python -m src.main train model=nemotron_nano train=qlora_t4 data.path={TRAIN_CSV}

# Bigger-GPU alternative (bf16 LoRA, rank 32) — only if an A100-class GPU is attached:
# !python -m src.main train model=nemotron_nano train=lora_a100 data.path={TRAIN_CSV}

## 6. (Optional) Evaluate boxed accuracy

Optional and **slow** (it generates with the fine-tuned adapter). Cap the sample count with
`data.max_samples=200` for a quick signal. `eval` reads the adapter from the same
`train.output_dir`, so pass the **same `train=` preset** you trained with.

In [ ]:
# Optional, slow — cap samples for a quick read. Use the same train= preset you trained with.
!python -m src.main eval model=nemotron_nano train=qlora_t4 data.path={TRAIN_CSV} data.max_samples=200

## 7. Package `submission.zip`

`package` zips the adapter at `cfg.train.output_dir` (flat) and validates `adapter_config.json`
with rank <= 32. Pass the **same `train=` preset** you trained with so it points at the right
adapter dir. Then copy the zip to `/kaggle/working/` so it's a **downloadable notebook output**.

In [ ]:
# Build submission.zip from the trained adapter (same train= preset as training/eval).
!python -m src.main package model=nemotron_nano train=qlora_t4

# Make it a downloadable Kaggle output.
!cp submission.zip /kaggle/working/submission.zip
!ls -lh /kaggle/working/submission.zip

## 8. Submit

This competition is graded on the **`submission.zip` LoRA adapter** (loaded into vLLM
server-side), not a predictions CSV.

**Option A — UI:** download `/kaggle/working/submission.zip` from the notebook **Output** tab,
then upload it on the competition's **Submit Predictions** page.

**Option B — Kaggle CLI** (needs API creds configured):

```bash
kaggle competitions submit \
    -c nvidia-nemotron-model-reasoning-challenge \
    -f /kaggle/working/submission.zip \
    -m "qlora_t4 LoRA adapter"
```

### Reminders the grader enforces
- LoRA **rank <= 32** (both presets comply; `package` re-checks `adapter_config.json`).
- The model must emit final answers in **`\boxed{...}`** and reasoning in **`<think>...</think>`**.